# How to improve concepts ?

With good concept, the potential increase in recall@1 is huge. Jumping from 33% to 73% ! -> We have to direct our efforts to better concept predition
Strategies :

    - A. Train longer on fine-tuned embeddings 
    - B. Train to predict elephant SEEK (instead of subject SEEK)
    - C. Tim’s take : Aggregate predicted subject SEEK into elephant SEEK at evaluation time. Is it realistic to do it in the Gallery ? in the query set ? Within the same encounter only in the query set ?

In [1]:
%load_ext autoreload
%autoreload 2

from seek_code import SEEK
from backbone import Backbone
from concept_head import ConceptHead
from retrieval import Retrieval
from projector import Projector
from data_handler import EleHandler
from concept_head_tunneled import ConceptHeadTunneled, ConceptHeadTunneledCategorical


from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
import torch.nn as nn


/data/vision/beery/scratch/antoine/micromamba/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
code = 'Improve_concept'

dataset = EleHandler(subset='IDI_6')
train_indices, test_indices = dataset.split_perpendicular_to_elephants_and_encounters(split_sizes=[0.5,0.5], hour_delta = 0.2)

train_subset = Subset(dataset, train_indices)
test_subset = Subset(dataset, test_indices)

train_loader = DataLoader(train_subset, batch_size=512, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=512, shuffle=False)

In [2]:

MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)

r = Retrieval(experiment_code=code)
r.evaluate_model(MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

NameError: name 'Backbone' is not defined

In [14]:
concept_head = ConceptHead(lr=0.001, loss=F.mse_loss, print_every=1, experiment_code = code)

MD_finetuned.freeze()
concept_head.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=50)
concept_head.test(test_loader, backbone = MD_finetuned)

r.evaluate_model(concept_head, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

Training ConceptHead for 50 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
Epoch 1/50 - Train Loss: 0.2770, Val Loss: 0.1936 - Train Acc: 38.54%, Val Acc: 54.48% - Train whole-code: 0.00%, Val whole-code: 0.00%
Epoch 2/50 - Train Loss: 0.1695, Val Loss: 0.1364 - Train Acc: 60.35%, Val Acc: 67.01% - Train whole-code: 0.21%, Val whole-code: 0.07%
Epoch 3/50 - Train Loss: 0.1241, Val Loss: 0.1160 - Train Acc: 70.12%, Val Acc: 71.35% - Train whole-code: 0.86%, Val whole-code: 1.03%
Epoch 4/50 - Train Loss: 0.1054, Val Loss: 0.1079 - Train Acc: 74.29%, Val Acc: 73.28% - Train whole-code: 2.57%, Val whole-code: 2.27%
Epoch 5/50 - Train Loss: 0.0962, Val Loss: 0.1041 - Train Acc: 76.28%, Val Acc: 74.43% - Train whole-code: 4.43%, Val whole-code: 3.71%
Epoch 6/50 - Train Loss: 0.0905, Val Loss: 0.1020 - Train Acc: 77.37%, Val Acc: 74.90% - Train whole-code: 5.94%, Val whole-code: 4.30%
Epoch 7/50 - Train Loss: 0.0864, Val Loss: 0.1008 - Trai

100%|██████████| 4/4 [00:13<00:00,  3.37s/it]


collecting concepts from concept head


100%|██████████| 3/3 [00:09<00:00,  3.17s/it]


Train Recalls:
Recall@1: 2.52%
Recall@5: 6.71%
Recall@10: 12.47%
Recall@20: 19.38%
Recall@100: 45.58%
Test Recalls:
Recall@1: 1.22%
Recall@5: 4.80%
Recall@10: 7.62%
Recall@20: 11.88%
Recall@100: 36.56%


## Let's try learning the actual ele_SEEK_code

In [15]:
code = 'Improve_concept'
MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
MD_finetuned.freeze()

r = Retrieval(experiment_code=code)

concept_head = ConceptHead(lr=0.001, loss=F.mse_loss, print_every=1, experiment_code = code)
concept_head.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=50, predict_ele_SEEK = True)
concept_head.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = True)

r.evaluate_model(concept_head, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

Training ConceptHead for 50 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
Epoch 1/50 - Train Loss: 0.2691, Val Loss: 0.1803 - Train Acc: 40.31%, Val Acc: 60.48% - Train whole-code: 0.00%, Val whole-code: 0.12%
Epoch 2/50 - Train Loss: 0.1554, Val Loss: 0.1205 - Train Acc: 65.81%, Val Acc: 72.72% - Train whole-code: 0.36%, Val whole-code: 0.49%
Epoch 3/50 - Train Loss: 0.1089, Val Loss: 0.1006 - Train Acc: 75.25%, Val Acc: 75.85% - Train whole-code: 1.27%, Val whole-code: 1.06%
Epoch 4/50 - Train Loss: 0.0909, Val Loss: 0.0934 - Train Acc: 78.62%, Val Acc: 76.94% - Train whole-code: 2.52%, Val whole-code: 1.26%
Epoch 5/50 - Train Loss: 0.0822, Val Loss: 0.0903 - Train Acc: 80.20%, Val Acc: 77.41% - Train whole-code: 3.23%, Val whole-code: 1.52%
Epoch 6/50 - Train Loss: 0.0767, Val Loss: 0.0887 - Train Acc: 81.21%, Val Acc: 77.54% - Train whole-code: 4.03%, Val whole-code: 1.75%
Epoch 7/50 - Train Loss: 0.0727, Val Loss: 0.0879 - Trai

100%|██████████| 4/4 [00:13<00:00,  3.38s/it]


collecting concepts from concept head


100%|██████████| 3/3 [00:07<00:00,  2.66s/it]


Train Recalls:
Recall@1: 42.20%
Recall@5: 58.20%
Recall@10: 66.68%
Recall@20: 76.78%
Recall@100: 91.52%
Test Recalls:
Recall@1: 1.98%
Recall@5: 4.57%
Recall@10: 6.47%
Recall@20: 8.91%
Recall@100: 22.85%


## Bigger Network

In [ ]:
MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
MD_finetuned.freeze()

r = Retrieval(experiment_code=code)

concept_head = ConceptHead(lr=0.001, loss=F.mse_loss, print_every=1, experiment_code = code)

concept_head.layer = nn.Sequential(
            nn.Linear(concept_head.input_dim, 200), 
            nn.LeakyReLU(),
            nn.Linear(200, 63), 
            nn.Sigmoid()
            ).to('cuda')

concept_head.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=50, predict_ele_SEEK = True)
concept_head.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = True)

r.evaluate_model(concept_head, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)


Training ConceptHead for 50 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
Epoch 1/50 - Train Loss: 0.2271, Val Loss: 0.1416 - Train Acc: 47.16%, Val Acc: 69.89% - Train whole-code: 0.06%, Val whole-code: 0.31%
Epoch 2/50 - Train Loss: 0.1152, Val Loss: 0.0980 - Train Acc: 74.67%, Val Acc: 76.46% - Train whole-code: 1.01%, Val whole-code: 1.01%
Epoch 3/50 - Train Loss: 0.0841, Val Loss: 0.0891 - Train Acc: 79.51%, Val Acc: 77.49% - Train whole-code: 2.78%, Val whole-code: 1.52%
Epoch 4/50 - Train Loss: 0.0754, Val Loss: 0.0874 - Train Acc: 81.17%, Val Acc: 77.77% - Train whole-code: 3.33%, Val whole-code: 0.90%
Epoch 5/50 - Train Loss: 0.0700, Val Loss: 0.0865 - Train Acc: 82.53%, Val Acc: 78.11% - Train whole-code: 4.33%, Val whole-code: 1.13%
Epoch 6/50 - Train Loss: 0.0652, Val Loss: 0.0858 - Train Acc: 83.86%, Val Acc: 78.13% - Train whole-code: 6.74%, Val whole-code: 1.76%
Epoch 7/50 - Train Loss: 0.0611, Val Loss: 0.0856 - Trai

100%|██████████| 4/4 [00:12<00:00,  3.17s/it]


collecting concepts from concept head


100%|██████████| 3/3 [00:08<00:00,  2.96s/it]


Train Recalls:
Recall@1: 63.05%
Recall@5: 73.65%
Recall@10: 80.01%
Recall@20: 89.35%
Recall@100: 98.28%
Test Recalls:
Recall@1: 2.51%
Recall@5: 3.81%
Recall@10: 4.65%
Recall@20: 6.17%
Recall@100: 18.51%


## Other loss

In [17]:
MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
MD_finetuned.freeze()

r = Retrieval(experiment_code=code)

concept_head = ConceptHead(lr=0.001, loss=nn.CrossEntropyLoss(), print_every=1, experiment_code = code)

concept_head.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=50, predict_ele_SEEK = True)
concept_head.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = True)

r.evaluate_model(concept_head, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

Training ConceptHead for 50 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
Epoch 1/50 - Train Loss: 65.6348, Val Loss: 63.2449 - Train Acc: 39.08%, Val Acc: 61.18% - Train whole-code: 0.00%, Val whole-code: 0.25%
Epoch 2/50 - Train Loss: 62.4951, Val Loss: 61.1918 - Train Acc: 66.27%, Val Acc: 73.54% - Train whole-code: 0.45%, Val whole-code: 0.56%
Epoch 3/50 - Train Loss: 60.7832, Val Loss: 60.2625 - Train Acc: 75.71%, Val Acc: 76.23% - Train whole-code: 1.43%, Val whole-code: 0.69%
Epoch 4/50 - Train Loss: 59.9482, Val Loss: 59.8460 - Train Acc: 78.63%, Val Acc: 77.13% - Train whole-code: 2.19%, Val whole-code: 0.69%
Epoch 5/50 - Train Loss: 59.5190, Val Loss: 59.6422 - Train Acc: 79.84%, Val Acc: 77.55% - Train whole-code: 3.04%, Val whole-code: 0.62%
Epoch 6/50 - Train Loss: 59.2669, Val Loss: 59.5318 - Train Acc: 80.49%, Val Acc: 77.88% - Train whole-code: 3.34%, Val whole-code: 0.82%
Epoch 7/50 - Train Loss: 59.0967, Val Loss: 

100%|██████████| 4/4 [00:13<00:00,  3.40s/it]


collecting concepts from concept head


100%|██████████| 3/3 [00:07<00:00,  2.64s/it]


Train Recalls:
Recall@1: 13.73%
Recall@5: 25.85%
Recall@10: 33.42%
Recall@20: 45.99%
Recall@100: 71.48%
Test Recalls:
Recall@1: 1.52%
Recall@5: 4.27%
Recall@10: 5.86%
Recall@20: 10.51%
Recall@100: 29.32%


## Ele-SEEK, Bigger network, dropout and cross-entropy

In [25]:
MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
MD_finetuned.freeze()

r = Retrieval(experiment_code=code)
big = nn.Sequential(
            nn.Linear(768 * 3, 256), 
            nn.Dropout(0.5),
            nn.LeakyReLU(),
            nn.Linear(256, 63)
            )

concept_head = ConceptHead(lr=0.001, loss=nn.CrossEntropyLoss(), print_every=1, experiment_code = code, architecture=big)

concept_head.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=50, predict_ele_SEEK = True)
concept_head.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = True)

r.evaluate_model(concept_head, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Training ConceptHead for 50 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
output shape torch.Size([512, 63]) labels shape torch.Size([512, 63])
output shape torch.Size([512, 63]) labels shape torch.Size([512, 63])
output shape torch.Size([512, 63]) labels shape torch.Size([512, 63])
output shape torch.Size([445, 63]) labels shape torch.Size([445, 63])


KeyboardInterrupt: 

## Back to subject SEEK-codes, learning with 3 networks, one for each ear and one for the general image

In [3]:
SEEK.test_separate_and_reconstruct()

B00T__E8000-0000X0_S00
Original one-hot :
 tensor([1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 0., 1., 0.,
        0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
        1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0.,
        0., 1., 0., 1., 0., 0., 1., 0., 0.])
Whole one-hot vector:
 tensor([[1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 1., 0., 1.,
         0., 0., 1., 0., 0.]])
Left one-hot vector:
 tensor([[0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0.,
         0., 0.]])
Right one-hot vector:
 tensor([[1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0.,
         0., 0.]])
reconstructed_one_hot tensor([1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1., 0., 1., 0.,
        0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
        1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0.,
        0., 1., 0., 1., 0., 0.

In [ ]:
from concept_head_tunneled import ConceptHeadTunneled



MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
MD_finetuned.freeze()

r = Retrieval(experiment_code=code)

small = [
        nn.Sequential(nn.Linear(768, 23)),
        nn.Sequential(nn.Linear(768, 20)),
        nn.Sequential(nn.Linear(768, 20))
]

concept_head_t = ConceptHeadTunneled(lr=0.001, loss=F.mse_loss, print_every=1, experiment_code = code, architecture = small)

concept_head_t.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=20, predict_ele_SEEK = False)

concept_head_t.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = False)

r.evaluate_model(concept_head_t, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)


/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Training ConceptHead for 20 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
Epoch 1/20 - Train Loss: 1.7041, Val Loss: 1.2560 - Train Acc: 31.39%, Val Acc: 43.52% - Train whole-code: 0.00%, Val whole-code: 0.00% - Train left ear: 27.19%, Val left ear: 32.68% - Train right ear: 21.05%, Val right ear: 32.59%
Epoch 2/20 - Train Loss: 1.0990, Val Loss: 1.0181 - Train Acc: 49.71%, Val Acc: 58.11% - Train whole-code: 0.00%, Val whole-code: 0.00% - Train left ear: 39.16%, Val left ear: 47.28% - Train right ear: 42.74%, Val right ear: 48.43%
Epoch 3/20 - Train Loss: 0.8621, Val Loss: 0.8795 - Train Acc: 59.57%, Val Acc: 60.21% - Train whole-code: 0.00%, Val whole-code: 0.00% - Train left ear: 49.63%, Val left ear: 48.74% - Train right ear: 51.16%, Val right ear: 49.10%
Epoch 4/20 - Train Loss: 0.7204, Val Loss: 0.7700 - Train Acc: 61.67%, Val Acc: 60.73% - Train whole-code: 0.00%, Val whole-code: 0.00% - Train left ear: 51.53%, Val left ear: 

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc_avg,▁▄▅▆▆▆▆▇▇▇▇▇▇███████
train_loss,█▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc_avg,▁▅▆▆▆▆▇▇▇▇██████████
val_loss,█▆▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch,20
train_acc_avg,0.78172
train_loss,0.28453
val_acc_avg,0.67906
val_loss,0.45419


evaluating concept head  
collecting concepts from concept head


100%|██████████| 4/4 [00:14<00:00,  3.58s/it]


collecting concepts from concept head


100%|██████████| 3/3 [00:09<00:00,  3.27s/it]


Train Recalls:
Recall@1: 0.86%
Recall@5: 3.89%
Recall@10: 6.71%
Recall@20: 11.11%
Recall@100: 34.28%
Test Recalls:
Recall@1: 0.84%
Recall@5: 2.89%
Recall@10: 4.27%
Recall@20: 8.83%
Recall@100: 33.82%


In [ ]:

MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
MD_finetuned.freeze()

r = Retrieval(experiment_code=code)

small_with_sigmmoid = [
                nn.Sequential(
                        nn.Linear(768, 23),
                        nn.Sigmoid()
                        ),
                nn.Sequential(
                        nn.Linear(768, 20),
                        nn.Sigmoid()
                        ),
                nn.Sequential(
                        nn.Linear(768, 20),
                        nn.Sigmoid()
                        )
]

concept_head_t = ConceptHeadTunneled(lr=0.001, loss=F.mse_loss, print_every=1, experiment_code = code, architecture = small_with_sigmmoid)


concept_head_t.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=20, predict_ele_SEEK = False)

concept_head_t.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = False)

r.evaluate_model(concept_head_t, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)


/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Training ConceptHead for 20 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
Epoch 1/20 - Train Loss: 0.7448, Val Loss: 0.6277 - Train Acc: 33.63%, Val Acc: 55.47% - Train whole-code: 0.00%, Val whole-code: 0.00% - Train left ear: 25.22%, Val left ear: 46.30% - Train right ear: 24.21%, Val right ear: 42.61%
Epoch 2/20 - Train Loss: 0.5961, Val Loss: 0.5306 - Train Acc: 58.86%, Val Acc: 64.44% - Train whole-code: 0.06%, Val whole-code: 0.44% - Train left ear: 49.65%, Val left ear: 52.44% - Train right ear: 48.19%, Val right ear: 52.90%
Epoch 3/20 - Train Loss: 0.5112, Val Loss: 0.4781 - Train Acc: 66.49%, Val Acc: 67.40% - Train whole-code: 0.30%, Val whole-code: 1.08% - Train left ear: 56.35%, Val left ear: 56.78% - Train right ear: 56.07%, Val right ear: 55.87%
Epoch 4/20 - Train Loss: 0.4602, Val Loss: 0.4454 - Train Acc: 69.34%, Val Acc: 68.64% - Train whole-code: 0.80%, Val whole-code: 0.57% - Train left ear: 59.68%, Val left ear: 

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc_avg,▁▅▆▇▇▇▇▇▇███████████
train_loss,█▆▄▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁
val_acc_avg,▁▅▇▇▇███████████████
val_loss,█▆▄▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁
epoch,20
train_acc_avg,0.76601
train_loss,0.27991
val_acc_avg,0.7043
val_loss,0.33668


evaluating concept head  
collecting concepts from concept head


100%|██████████| 4/4 [00:12<00:00,  3.20s/it]


collecting concepts from concept head


100%|██████████| 3/3 [00:08<00:00,  2.68s/it]


Train Recalls:
Recall@1: 1.01%
Recall@5: 3.13%
Recall@10: 5.45%
Recall@20: 10.10%
Recall@100: 31.10%
Test Recalls:
Recall@1: 0.69%
Recall@5: 2.97%
Recall@10: 5.18%
Recall@20: 9.75%
Recall@100: 32.98%


## Bigger networks, with dropout

In [4]:
from concept_head_tunneled import ConceptHeadTunneled

MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
MD_finetuned.freeze()


r = Retrieval(experiment_code=code)

bigger = [
                nn.Sequential(
                        nn.Linear(768, 256),
                        nn.LeakyReLU(),
                        nn.Dropout(0.5),
                        nn.Linear(256, 23)
                        ),
                nn.Sequential(
                        nn.Linear(768, 256),
                        nn.LeakyReLU(),
                        nn.Dropout(0.5),
                        nn.Linear(256, 20)
                        ),
                nn.Sequential(
                        nn.Linear(768, 256),
                        nn.LeakyReLU(),
                        nn.Dropout(0.5),
                        nn.Linear(256, 20)
                        )
]

concept_head_t = ConceptHeadTunneled(lr=0.001, loss=F.mse_loss, print_every=1, experiment_code = code, architecture = bigger)

concept_head_t.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=20, predict_ele_SEEK = False)
concept_head_t.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = False)

r.evaluate_model(concept_head_t, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Training ConceptHead for 20 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
Epoch 1/20 - Train Loss: 0.9307, Val Loss: 0.4792 - Train Acc: 38.97%, Val Acc: 65.99% - Train whole-code: 0.00%, Val whole-code: 0.43% - Train left ear: 35.34%, Val left ear: 54.08% - Train right ear: 37.51%, Val right ear: 54.78%
Epoch 2/20 - Train Loss: 0.5649, Val Loss: 0.3972 - Train Acc: 63.52%, Val Acc: 67.94% - Train whole-code: 0.05%, Val whole-code: 0.43% - Train left ear: 52.90%, Val left ear: 56.84% - Train right ear: 54.24%, Val right ear: 55.95%
Epoch 3/20 - Train Loss: 0.4395, Val Loss: 0.3723 - Train Acc: 67.93%, Val Acc: 68.44% - Train whole-code: 0.50%, Val whole-code: 0.43% - Train left ear: 57.65%, Val left ear: 57.75% - Train right ear: 58.04%, Val right ear: 56.60%
Epoch 4/20 - Train Loss: 0.3985, Val Loss: 0.3712 - Train Acc: 69.97%, Val Acc: 69.08% - Train whole-code: 0.97%, Val whole-code: 1.23% - Train left ear: 60.60%, Val left ear: 

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc_avg,▁▅▅▆▆▆▇▇▇▇▇▇▇███████
train_loss,█▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_acc_avg,▁▄▅▅▆▇▇▇▇▇▇█████████
val_loss,█▄▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,20
train_acc_avg,0.84047
train_loss,0.21926
val_acc_avg,0.70609
val_loss,0.33042


evaluating concept head  
collecting concepts from concept head


100%|██████████| 4/4 [00:13<00:00,  3.40s/it]


collecting concepts from concept head


100%|██████████| 3/3 [00:08<00:00,  2.78s/it]


Train Recalls:
Recall@1: 0.35%
Recall@5: 3.43%
Recall@10: 5.25%
Recall@20: 9.09%
Recall@100: 28.52%
Test Recalls:
Recall@1: 1.29%
Recall@5: 2.89%
Recall@10: 4.72%
Recall@20: 9.06%
Recall@100: 31.15%


In [5]:
MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
MD_finetuned.freeze()


r = Retrieval(experiment_code=code)

bigger_with_sigmoid = [
                nn.Sequential(
                        nn.Linear(768, 256),
                        nn.LeakyReLU(),
                        nn.Dropout(0.5),
                        nn.Linear(256, 23),
                        nn.Sigmoid()
                        ),
                nn.Sequential(
                        nn.Linear(768, 256),
                        nn.LeakyReLU(),
                        nn.Dropout(0.5),
                        nn.Linear(256, 20),
                        nn.Sigmoid()
                        ),
                nn.Sequential(
                        nn.Linear(768, 256),
                        nn.LeakyReLU(),
                        nn.Dropout(0.5),
                        nn.Linear(256, 20),
                        nn.Sigmoid()
                        )
]

concept_head_t = ConceptHeadTunneled(lr=0.001, loss=F.mse_loss, print_every=1, experiment_code = code, architecture = bigger_with_sigmoid)

concept_head_t.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=20, predict_ele_SEEK = False)
concept_head_t.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = False)

r.evaluate_model(concept_head_t, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Training ConceptHead for 20 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
Epoch 1/20 - Train Loss: 0.6977, Val Loss: 0.5462 - Train Acc: 43.47%, Val Acc: 65.76% - Train whole-code: 0.00%, Val whole-code: 0.07% - Train left ear: 30.89%, Val left ear: 53.53% - Train right ear: 34.82%, Val right ear: 52.24%
Epoch 2/20 - Train Loss: 0.5154, Val Loss: 0.4389 - Train Acc: 64.67%, Val Acc: 68.39% - Train whole-code: 0.05%, Val whole-code: 0.20% - Train left ear: 50.55%, Val left ear: 57.18% - Train right ear: 52.40%, Val right ear: 56.80%
Epoch 3/20 - Train Loss: 0.4254, Val Loss: 0.3897 - Train Acc: 68.52%, Val Acc: 69.22% - Train whole-code: 0.20%, Val whole-code: 0.80% - Train left ear: 56.25%, Val left ear: 58.43% - Train right ear: 57.72%, Val right ear: 58.23%
Epoch 4/20 - Train Loss: 0.3747, Val Loss: 0.3568 - Train Acc: 70.58%, Val Acc: 69.80% - Train whole-code: 0.76%, Val whole-code: 1.44% - Train left ear: 59.65%, Val left ear: 

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc_avg,▁▅▆▆▆▆▆▇▇▇▇▇▇▇██████
train_loss,█▅▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
val_acc_avg,▁▄▅▆▆▇▇▇████████████
val_loss,█▅▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,20
train_acc_avg,0.81528
train_loss,0.21558
val_acc_avg,0.71282
val_loss,0.31298


evaluating concept head  
collecting concepts from concept head


100%|██████████| 4/4 [00:12<00:00,  3.13s/it]


collecting concepts from concept head


100%|██████████| 3/3 [00:08<00:00,  2.75s/it]


Train Recalls:
Recall@1: 1.31%
Recall@5: 3.89%
Recall@10: 5.81%
Recall@20: 9.64%
Recall@100: 28.27%
Test Recalls:
Recall@1: 0.76%
Recall@5: 2.59%
Recall@10: 4.49%
Recall@20: 7.92%
Recall@100: 29.93%


In [6]:
from concept_head_tunneled import ConceptHeadTunneled

MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
MD_finetuned.freeze()


r = Retrieval(experiment_code=code)

bigger = [
                nn.Sequential(
                        nn.Linear(768, 256),
                        nn.LeakyReLU(),
                        nn.Dropout(0.5),
                        nn.Linear(256, 23)
                        ),
                nn.Sequential(
                        nn.Linear(768, 256),
                        nn.LeakyReLU(),
                        nn.Dropout(0.5),
                        nn.Linear(256, 20)
                        ),
                nn.Sequential(
                        nn.Linear(768, 256),
                        nn.LeakyReLU(),
                        nn.Dropout(0.5),
                        nn.Linear(256, 20)
                        )
]

concept_head_t = ConceptHeadTunneled(lr=0.001, loss=nn.CrossEntropyLoss(), print_every=1, experiment_code = code, architecture = bigger)

concept_head_t.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=20, predict_ele_SEEK = False)
concept_head_t.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = False)

r.evaluate_model(concept_head_t, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Training ConceptHead for 20 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
Epoch 1/20 - Train Loss: 47.1654, Val Loss: 43.4494 - Train Acc: 49.12%, Val Acc: 67.57% - Train whole-code: 0.00%, Val whole-code: 0.49% - Train left ear: 37.91%, Val left ear: 57.63% - Train right ear: 36.85%, Val right ear: 53.81%
Epoch 2/20 - Train Loss: 43.2836, Val Loss: 42.2732 - Train Acc: 68.66%, Val Acc: 69.29% - Train whole-code: 0.41%, Val whole-code: 0.95% - Train left ear: 58.24%, Val left ear: 59.15% - Train right ear: 56.65%, Val right ear: 57.74%
Epoch 3/20 - Train Loss: 41.9518, Val Loss: 41.4909 - Train Acc: 70.71%, Val Acc: 69.80% - Train whole-code: 1.12%, Val whole-code: 2.12% - Train left ear: 61.26%, Val left ear: 59.94% - Train right ear: 59.69%, Val right ear: 58.56%
Epoch 4/20 - Train Loss: 40.8392, Val Loss: 40.7525 - Train Acc: 72.34%, Val Acc: 70.11% - Train whole-code: 1.78%, Val whole-code: 2.45% - Train left ear: 63.01%, Val le

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc_avg,▁▅▅▆▆▆▆▆▇▇▇▇▇▇▇█████
train_loss,█▆▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
val_acc_avg,▁▄▅▆▆▇▇▇████████████
val_loss,█▅▄▂▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂
epoch,20
train_acc_avg,0.84332
train_loss,35.33493
val_acc_avg,0.71066
val_loss,40.81777


evaluating concept head  
collecting concepts from concept head


100%|██████████| 4/4 [00:13<00:00,  3.28s/it]


collecting concepts from concept head


100%|██████████| 3/3 [00:08<00:00,  2.68s/it]


Train Recalls:
Recall@1: 1.01%
Recall@5: 3.53%
Recall@10: 5.50%
Recall@20: 9.74%
Recall@100: 30.99%
Test Recalls:
Recall@1: 0.91%
Recall@5: 2.82%
Recall@10: 4.42%
Recall@20: 7.62%
Recall@100: 30.08%


## I would want to try concatenating the ear embeddings with the general embeddings to predict the ears. 

In [7]:
bigger = [
                nn.Sequential(
                        nn.Linear(768, 256),
                        nn.LeakyReLU(),
                        nn.Dropout(0.6),
                        nn.Linear(256, 23)
                        ),
                nn.Sequential(
                        nn.Linear(768, 256),
                        nn.LeakyReLU(),
                        nn.Dropout(0.6),
                        nn.Linear(256, 20)
                        ),
                nn.Sequential(
                        nn.Linear(768, 256),
                        nn.LeakyReLU(),
                        nn.Dropout(0.6),
                        nn.Linear(256, 20)
                        )
]
from torch.optim import Adam

concept_head_t = ConceptHeadTunneled(lr=0.001, loss=nn.CrossEntropyLoss(), print_every=1, experiment_code = code, architecture = bigger)

concept_head_t.left_ear_optimizer = Adam(concept_head_t.left_ear_layer.parameters(), lr=1e-3, weight_decay=1e-4)
concept_head_t.right_ear_optimizer = Adam(concept_head_t.right_ear_layer.parameters(), lr=1e-3, weight_decay=1e-4)
concept_head_t.whole_image_optimizer = Adam(concept_head_t.whole_image_layer.parameters(), lr=1e-3, weight_decay=1e-4)

concept_head_t.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=20, predict_ele_SEEK = False)
concept_head_t.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = False)

r.evaluate_model(concept_head_t, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

Training ConceptHead for 20 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
Epoch 1/20 - Train Loss: 48.1341, Val Loss: 43.9286 - Train Acc: 44.51%, Val Acc: 67.41% - Train whole-code: 0.00%, Val whole-code: 0.07% - Train left ear: 38.37%, Val left ear: 55.12% - Train right ear: 34.99%, Val right ear: 56.26%
Epoch 2/20 - Train Loss: 43.9334, Val Loss: 42.3004 - Train Acc: 67.32%, Val Acc: 69.08% - Train whole-code: 0.20%, Val whole-code: 1.47% - Train left ear: 56.14%, Val left ear: 58.57% - Train right ear: 56.42%, Val right ear: 57.67%
Epoch 3/20 - Train Loss: 42.4791, Val Loss: 41.3931 - Train Acc: 69.90%, Val Acc: 69.83% - Train whole-code: 0.76%, Val whole-code: 1.34% - Train left ear: 58.65%, Val left ear: 59.91% - Train right ear: 59.92%, Val right ear: 58.71%
Epoch 4/20 - Train Loss: 41.2442, Val Loss: 40.7218 - Train Acc: 71.38%, Val Acc: 70.28% - Train whole-code: 1.22%, Val whole-code: 2.04% - Train left ear: 61.48%, Val le

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc_avg,▁▅▆▆▆▆▇▇▇▇▇▇▇▇██████
train_loss,█▆▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
val_acc_avg,▁▄▆▆▇▇▇▇████████████
val_loss,█▅▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂
epoch,20
train_acc_avg,0.82762
train_loss,35.9252
val_acc_avg,0.71005
val_loss,40.63028


evaluating concept head  
collecting concepts from concept head


100%|██████████| 4/4 [00:12<00:00,  3.17s/it]


collecting concepts from concept head


100%|██████████| 3/3 [00:08<00:00,  2.74s/it]


Train Recalls:
Recall@1: 1.26%
Recall@5: 3.53%
Recall@10: 5.75%
Recall@20: 9.89%
Recall@100: 29.88%
Test Recalls:
Recall@1: 1.14%
Recall@5: 2.74%
Recall@10: 4.11%
Recall@20: 8.53%
Recall@100: 29.86%


In [13]:
bigger = [
                nn.Sequential(
                        nn.Linear(768, 512),
                        nn.LeakyReLU(),
                        nn.Linear(512, 256),
                        nn.LeakyReLU(),
                        nn.Linear(256, 128),
                        nn.LeakyReLU(),
                        nn.Dropout(0.4),
                        nn.LeakyReLU(),
                        nn.Linear(128, 23)
                        ),
                nn.Sequential(
                        nn.Linear(768, 512),
                        nn.LeakyReLU(),
                        nn.Linear(512, 256),
                        nn.LeakyReLU(),
                        nn.Linear(256, 128),
                        nn.LeakyReLU(),
                        nn.Dropout(0.4),
                        nn.LeakyReLU(),
                        nn.Linear(128, 20)
                        ),
                nn.Sequential(
                        nn.Linear(768, 512),
                        nn.LeakyReLU(),
                        nn.Linear(512, 256),
                        nn.LeakyReLU(),
                        nn.Linear(256, 128),
                        nn.LeakyReLU(),
                        nn.Dropout(0.4),
                        nn.LeakyReLU(),
                        nn.Linear(128, 20)
                        )
]
from torch.optim import Adam

concept_head_t = ConceptHeadTunneled(lr=0.001, loss=nn.CrossEntropyLoss(), print_every=1, experiment_code = code, architecture = bigger)

concept_head_t.left_ear_optimizer = Adam(concept_head_t.left_ear_layer.parameters(), lr=1e-3, weight_decay=1e-4)
concept_head_t.right_ear_optimizer = Adam(concept_head_t.right_ear_layer.parameters(), lr=1e-3, weight_decay=1e-4)
concept_head_t.whole_image_optimizer = Adam(concept_head_t.whole_image_layer.parameters(), lr=1e-3, weight_decay=1e-4)

concept_head_t.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=20, predict_ele_SEEK = False)
concept_head_t.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = False)

r.evaluate_model(concept_head_t, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

Training ConceptHead for 20 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
Epoch 1/20 - Train Loss: 47.3641, Val Loss: 44.2413 - Train Acc: 50.90%, Val Acc: 60.30% - Train whole-code: 0.00%, Val whole-code: 0.00% - Train left ear: 41.29%, Val left ear: 53.40% - Train right ear: 29.92%, Val right ear: 34.72%
Epoch 2/20 - Train Loss: 43.9905, Val Loss: 42.3618 - Train Acc: 61.43%, Val Acc: 66.69% - Train whole-code: 0.00%, Val whole-code: 0.00% - Train left ear: 51.20%, Val left ear: 55.27% - Train right ear: 40.80%, Val right ear: 53.29%
Epoch 3/20 - Train Loss: 42.4491, Val Loss: 41.3219 - Train Acc: 65.76%, Val Acc: 68.02% - Train whole-code: 0.06%, Val whole-code: 0.00% - Train left ear: 53.22%, Val left ear: 57.99% - Train right ear: 52.02%, Val right ear: 54.83%
Epoch 4/20 - Train Loss: 41.2767, Val Loss: 40.8717 - Train Acc: 68.45%, Val Acc: 69.73% - Train whole-code: 0.43%, Val whole-code: 1.83% - Train left ear: 57.61%, Val le

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc_avg,▁▃▄▅▅▅▆▆▇▇▇▇▇▇██████
train_loss,█▆▅▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁
val_acc_avg,▁▅▆▇▇▇██████████████
val_loss,█▅▃▂▁▁▁▁▂▂▃▃▃▃▃▄▄▄▅▅
epoch,20
train_acc_avg,0.84941
train_loss,34.83084
val_acc_avg,0.70431
val_loss,42.80052


evaluating concept head  
collecting concepts from concept head


100%|██████████| 4/4 [00:12<00:00,  3.03s/it]


collecting concepts from concept head


100%|██████████| 3/3 [00:07<00:00,  2.66s/it]


Train Recalls:
Recall@1: 0.61%
Recall@5: 2.93%
Recall@10: 5.30%
Recall@20: 8.73%
Recall@100: 28.62%
Test Recalls:
Recall@1: 0.53%
Recall@5: 2.67%
Recall@10: 4.57%
Recall@20: 7.46%
Recall@100: 27.19%


In [ ]:
Train Recalls:
Recall@1: 0.61%
Recall@5: 2.93%
Recall@10: 5.30%
Recall@20: 8.73%
Recall@100: 28.62%
Test Recalls:
Recall@1: 0.53%
Recall@5: 2.67%
Recall@10: 4.57%
Recall@20: 7.46%
Recall@100: 27.19%

In [14]:
bigger = [
                nn.Sequential(
                        nn.Linear(768, 512),
                        nn.LeakyReLU(),
                        nn.Linear(512, 256),
                        nn.LeakyReLU(),
                        nn.Linear(256, 128),
                        nn.LeakyReLU(),
                        nn.Dropout(0.4),
                        nn.LeakyReLU(),
                        nn.Linear(128, 23)
                        ),
                nn.Sequential(
                        nn.Linear(768, 512),
                        nn.LeakyReLU(),
                        nn.Linear(512, 256),
                        nn.LeakyReLU(),
                        nn.Linear(256, 128),
                        nn.LeakyReLU(),
                        nn.Dropout(0.4),
                        nn.LeakyReLU(),
                        nn.Linear(128, 20)
                        ),
                nn.Sequential(
                        nn.Linear(768, 512),
                        nn.LeakyReLU(),
                        nn.Linear(512, 256),
                        nn.LeakyReLU(),
                        nn.Linear(256, 128),
                        nn.LeakyReLU(),
                        nn.Dropout(0.4),
                        nn.LeakyReLU(),
                        nn.Linear(128, 20)
                        )
]
from torch.optim import Adam

concept_head_t = ConceptHeadTunneled(lr=0.001, loss=nn.CrossEntropyLoss(), print_every=1, experiment_code = code, architecture = bigger)

concept_head_t.left_ear_optimizer = Adam(concept_head_t.left_ear_layer.parameters(), lr=1e-3, weight_decay=1e-4)
concept_head_t.right_ear_optimizer = Adam(concept_head_t.right_ear_layer.parameters(), lr=1e-3, weight_decay=1e-4)
concept_head_t.whole_image_optimizer = Adam(concept_head_t.whole_image_layer.parameters(), lr=1e-3, weight_decay=1e-4)

concept_head_t.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=20, predict_ele_SEEK = True)
concept_head_t.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = True)

r.evaluate_model(concept_head_t, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

Training ConceptHead for 20 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
Epoch 1/20 - Train Loss: 47.0433, Val Loss: 42.1732 - Train Acc: 53.90%, Val Acc: 72.11% - Train whole-code: 0.00%, Val whole-code: 0.64% - Train left ear: 29.74%, Val left ear: 53.25% - Train right ear: 48.93%, Val right ear: 65.64%
Epoch 2/20 - Train Loss: 42.2829, Val Loss: 39.5577 - Train Acc: 74.45%, Val Acc: 76.85% - Train whole-code: 0.20%, Val whole-code: 0.46% - Train left ear: 61.10%, Val left ear: 67.42% - Train right ear: 65.54%, Val right ear: 66.64%
Epoch 3/20 - Train Loss: 39.8580, Val Loss: 37.9979 - Train Acc: 76.89%, Val Acc: 77.04% - Train whole-code: 0.89%, Val whole-code: 0.52% - Train left ear: 67.99%, Val left ear: 67.32% - Train right ear: 66.20%, Val right ear: 67.34%
Epoch 4/20 - Train Loss: 38.4343, Val Loss: 37.7111 - Train Acc: 76.77%, Val Acc: 77.02% - Train whole-code: 0.67%, Val whole-code: 0.46% - Train left ear: 67.58%, Val le

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc_avg,▁▅▆▆▆▆▆▆▆▇▇▇▇▇▇█████
train_loss,█▅▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
val_acc_avg,▁▇▇▇▇███████████████
val_loss,█▅▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂
epoch,20
train_acc_avg,0.87266
train_loss,33.73548
val_acc_avg,0.77563
val_loss,37.43556


evaluating concept head  
collecting concepts from concept head


100%|██████████| 4/4 [00:13<00:00,  3.25s/it]


collecting concepts from concept head


100%|██████████| 3/3 [00:08<00:00,  2.83s/it]


Train Recalls:
Recall@1: 10.05%
Recall@5: 18.98%
Recall@10: 27.66%
Recall@20: 40.69%
Recall@100: 77.39%
Test Recalls:
Recall@1: 1.29%
Recall@5: 1.83%
Recall@10: 3.12%
Recall@20: 5.33%
Recall@100: 18.51%


## It would be interesting to switch to categorical data to see if it performs better

In [3]:
MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)


/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

In [7]:
concept_cat = ConceptHeadTunneledCategorical(lr=0.001, loss=nn.CrossEntropyLoss(), print_every=1, experiment_code = code)

concept_cat.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=20, predict_ele_SEEK = False)
concept_cat.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = False)

r.evaluate_model(concept_cat, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)

Training ConceptHead for 20 epochs, reseting weights, and the backbone parameters are, False  concept head parameters are  True
redicted[0] B20T11E_7_0-__53X01S1_
labels[0] B00T11E9000-0000X0_S00
redicted[0] C00T01E8887-0_30X_1S01
labels[0] B00T11E____-3050X_0S00
redicted[0] B00T_0E_087-0303X01S1_
labels[0] B00T11E9090-4030X00S00
redicted[0] C00T01E8890-_55_X0_S1_
labels[0] B00T1_E8090-4000X00S00
redicted[0] C20T11E8787-5543X00S01
labels[0] B00T11E9000-0000X00S00
redicted[0] B00T1_E9808-4_4_X0_S00
labels[0] B00T11E9090-0000X00S00
redicted[0] B00T01E9888-_4_3X__S10
labels[0] C20T11E____-0305X_0S0_
Epoch 1/20 - Train Loss: 1.7064, Val Loss: 1.2502 - Train Acc: 31.81%, Val Acc: 35.84% - Train whole-code: 0.00%, Val whole-code: 0.00% - Train left ear: 25.75%, Val left ear: 24.57% - Train right ear: 26.30%, Val right ear: 26.88%
redicted[0] B20T_1E_0_0-_553X01S0_
labels[0] B00T11E9000-0000X0_S00
redicted[0] C00T01E9988-0_30X_1S0_
labels[0] B00T11E____-3050X_0S00
redicted[0] B00T_0E__8_-4_05

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc_avg,▁▄▄▅▅▆▆▆▇▇▇▇████████
train_loss,█▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc_avg,▁▂▄▅▆▆▆▆▇▇▇▇████████
val_loss,█▆▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch,20
train_acc_avg,0.43857
train_loss,0.28303
val_acc_avg,0.42135
val_loss,0.43174


NameError: name 'r' is not defined